# Exploración de los datos de BiciBCN

Usad este notebook para conocer los datos **antes** de escribir `SPECS.md`. Lo que encontréis va a la sección **Entradas** de `SPECS.md`, y las decisiones a **Decisiones**.

Trabajamos con pandas. Para ficheros grandes, polars es más rápido (`pl.read_csv`, misma lógica).

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

DATOS = Path("../data/raw")
tablas = {n: pd.read_csv(DATOS / f"{n}.csv") for n in ['estaciones', 'barrios', 'distritos', 'viajes', 'estado_estaciones', 'redistribucion', 'abonados', 'meteo', 'festivos']}
for nombre, df in tablas.items():
    print(f"{nombre:20s} {df.shape[0]:>10,d} filas  {df.shape[1]:>2d} columnas")

## 1. Perfil de cada tabla

In [ ]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
    """Tipo, nulos, valores distintos y un ejemplo de cada columna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(1),
        "distintos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

perfil(tablas['estaciones'])

## 2. Preguntas de calidad para empezar

- ¿Es `viaje_id` único?
- ¿Qué valores toma `tipo_bici`?
- ¿Cómo se distribuye la duración de los viajes? ¿Hay negativos, nulos o viajes de días?
- ¿Todas las estaciones de `viajes` existen en `estaciones`?
- ¿Todas las estaciones están en Barcelona? ¿Qué altitudes y capacidades hay?
- ¿Coincide el distrito de cada estación con el de su barrio?
- En `estado_estaciones`: ¿puede haber más bicis que anclajes? ¿Hay estaciones que no cambian nunca?
- ¿Hay días sin viajes?

**Ejemplo:** duración de los viajes en minutos

In [ ]:
viajes = tablas["viajes"]
inicio = pd.to_datetime(viajes["fecha_hora_inicio"])
fin = pd.to_datetime(viajes["fecha_hora_fin"])
((fin - inicio).dt.total_seconds() / 60).describe()

**Ejemplo:** destinos que no están en el catálogo

In [ ]:
catalogo = set(tablas["estaciones"]["estacion_id"])
viajes.loc[~viajes["estacion_destino"].isin(catalogo), "estacion_destino"].value_counts(dropna=False)

In [ ]:
# Vuestras comprobaciones aquí


## 3. Una serie diaria: ¿algo raro?

Una serie por día es la forma más rápida de ver huecos y picos imposibles.

In [ ]:
diario = viajes.groupby(inicio.dt.date).size().rename("viajes_muestra")
px.line(diario, title="Viajes de la muestra por día")